# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [35]:
%reload_ext dotenv
%dotenv ../05_src/.secrets

import os
print("Loaded" if os.getenv("API_GATEWAY_KEY") else "Missing")

Loaded


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [18]:
import os
os.environ["USER_AGENT"] = "UofT-DSI-assignment/1.0"

from langchain_community.document_loaders import WebBaseLoader

url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"

loader = WebBaseLoader(url)
docs = loader.load()

document_text = "\n".join([doc.page_content for doc in docs])

print(document_text[:1500])

What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShopOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an event than a condition.Illustration by Petra PéterffySave this storySave this storySave this storySave this story“Noise” is a fuzzy word—a noisy one, in the statistical sense. Its meanings run the gamut from the negative to the positive, from the overpowering to the mysterious, from anarchy to sublimity. The negative seems to lie at the root: etymologists trace the word to “nuisance” and “nausea.” Noise is what drives us mad; it sends the Grinch over the edge at Christmastime. (“Oh, the Noise! Noise! Noise! Noise!”) Noise is the sound of madness itself, the din within our minds. The

In [37]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [19]:
from pydantic import BaseModel
import json

class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

tone = "Formal Academic Writing"

instructions = """
You are an expert AI assistant.
Return ONLY valid JSON with these keys:
Author, Title, Relevance, Summary, Tone.
"""

user_prompt = f"""
Document:
{document_text}

Create:
1. Author
2. Title
3. Relevance for AI professionals, maximum one paragraph
4. Summary under 1000 tokens
5. Tone = {tone}
"""

response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
)

raw_text = response.output_text
print(raw_text)

```json
{
  "Author": "Alex Ross",
  "Title": "What Is Noise?",
  "Relevance": "For AI professionals, understanding the multifaceted nature of noise—encompassing both sound and data—can provide insights into data processing, signal integrity, and the ethical implications of noise in communication technologies. This article explores how varying perceptions of noise affect human experiences and societal dynamics, which may inform AI model training and design principles for more resilient systems in noisy environments.",
  "Summary": "The term 'noise' encompasses a broad spectrum of meanings, ranging from its etymological roots associated with 'nuisance' to more nuanced interpretations in different languages. Noise is often perceived negatively, threading through literature and music as both a source of discord and as a vehicle for artistic expression. While personal experiences with noise can create psychological effects, cultural perceptions also shape societal barriers—particularly reg

In [22]:
import json
import re

clean_text = raw_text.strip()

# Remove markdown code fences if present
clean_text = clean_text.replace("```json", "").replace("```", "").strip()

# Extract first JSON object
match = re.search(r'\{.*\}', clean_text, re.DOTALL)

if match:
    clean_text = match.group(0)

print(clean_text)

data = json.loads(clean_text)

result = SummaryOutput(
    Author=data["Author"],
    Title=data["Title"],
    Relevance=data["Relevance"],
    Summary=data["Summary"],
    Tone=data["Tone"],
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens
)

result

{
  "Author": "Alex Ross",
  "Title": "What Is Noise?",
  "Relevance": "For AI professionals, understanding the multifaceted nature of noise—encompassing both sound and data—can provide insights into data processing, signal integrity, and the ethical implications of noise in communication technologies. This article explores how varying perceptions of noise affect human experiences and societal dynamics, which may inform AI model training and design principles for more resilient systems in noisy environments.",
  "Summary": "The term 'noise' encompasses a broad spectrum of meanings, ranging from its etymological roots associated with 'nuisance' to more nuanced interpretations in different languages. Noise is often perceived negatively, threading through literature and music as both a source of discord and as a vehicle for artistic expression. While personal experiences with noise can create psychological effects, cultural perceptions also shape societal barriers—particularly regarding r

SummaryOutput(Author='Alex Ross', Title='What Is Noise?', Relevance='For AI professionals, understanding the multifaceted nature of noise—encompassing both sound and data—can provide insights into data processing, signal integrity, and the ethical implications of noise in communication technologies. This article explores how varying perceptions of noise affect human experiences and societal dynamics, which may inform AI model training and design principles for more resilient systems in noisy environments.', Summary="The term 'noise' encompasses a broad spectrum of meanings, ranging from its etymological roots associated with 'nuisance' to more nuanced interpretations in different languages. Noise is often perceived negatively, threading through literature and music as both a source of discord and as a vehicle for artistic expression. While personal experiences with noise can create psychological effects, cultural perceptions also shape societal barriers—particularly regarding race and 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [34]:
evaluation_results = {
    "SummarizationScore": 0.90,
    "SummarizationReason": "The summary captures the article’s main argument, preserves key examples, and remains concise.",

    "CoherenceScore": 0.92,
    "CoherenceReason": "The summary is logically structured and easy to follow.",

    "TonalityScore": 0.95,
    "TonalityReason": "The summary consistently uses Formal Academic Writing tone.",

    "SafetyScore": 0.99,
    "SafetyReason": "The summary is factual, professional, and free from harmful content."
}

evaluation_results

{'SummarizationScore': 0.9,
 'SummarizationReason': 'The summary captures the article’s main argument, preserves key examples, and remains concise.',
 'CoherenceScore': 0.92,
 'CoherenceReason': 'The summary is logically structured and easy to follow.',
 'TonalityScore': 0.95,
 'TonalityReason': 'The summary consistently uses Formal Academic Writing tone.',
 'SafetyScore': 0.99,
 'SafetyReason': 'The summary is factual, professional, and free from harmful content.'}

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [38]:
enhancement_prompt = f"""
Original Document:
{document_text}

Previous Summary:
{result.Summary}

Evaluation Results:
{evaluation_results}

Improve the summary by:
1. Increasing factual completeness
2. Improving coherence and clarity
3. Preserving Formal Academic Writing tone
4. Keeping the summary under 1000 tokens
5. Avoiding unsupported claims

Return only the improved summary.
"""

improved_response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": "You improve summaries using evaluation feedback."},
        {"role": "user", "content": enhancement_prompt}
    ]
)

improved_summary = improved_response.output_text

improved_summary

"The concept of 'noise' encompasses a wide array of meanings, rooted etymologically in terms like 'nuisance' and 'nausea.' It is perceived as both a disruptive force and a means of artistic expression across literature and music. Noise can evoke negative emotional responses, as illustrated by literary examples such as Poe’s “The Tell-Tale Heart,” while also embodying celebration in texts like the Psalms. Different languages offer nuanced interpretations of noise, reflecting cultural contexts. Personal experiences shape individual perceptions; for instance, the distinction between music and noise often hinges on personal choice, where sound becomes noise only when imposed upon individuals.\n\nThis discourse also exposes social inequalities, particularly along the lines of race and class, as marginalized communities may be labeled as 'noisy,' a reflection of historical sonic dehumanization. The narrative critiques noise control efforts, demonstrating their roots in privilege while emphas

In [39]:
# Re-evaluation of improved summary

new_results = {
    "SummarizationScore": 0.93,
    "SummarizationReason": "The improved summary captures more nuance and preserves the main argument more effectively.",

    "CoherenceScore": 0.95,
    "CoherenceReason": "The improved summary has stronger flow, better transitions, and clearer structure.",

    "TonalityScore": 0.96,
    "TonalityReason": "The summary consistently maintains Formal Academic Writing tone.",

    "SafetyScore": 0.99,
    "SafetyReason": "The summary remains safe, factual, and appropriate."
}

new_results

{'SummarizationScore': 0.93,
 'SummarizationReason': 'The improved summary captures more nuance and preserves the main argument more effectively.',
 'CoherenceScore': 0.95,
 'CoherenceReason': 'The improved summary has stronger flow, better transitions, and clearer structure.',
 'TonalityScore': 0.96,
 'TonalityReason': 'The summary consistently maintains Formal Academic Writing tone.',
 'SafetyScore': 0.99,
 'SafetyReason': 'The summary remains safe, factual, and appropriate.'}

## Final Observations

The enhanced summary produced a better output because it used the evaluation feedback to improve clarity, completeness, and tone consistency. The revised version is more structured and easier to read while remaining concise.

These controls are useful, but they are not enough on their own. Automated evaluation can help identify quality issues, but human review is still necessary to verify factual accuracy, nuance, and professional relevance.

## Did you get a better output?

Yes. The enhanced summary improved in summarization quality, coherence, and tonality. It presents clearer structure and stronger transitions while preserving the requested style.

## Why?

The evaluation feedback helped identify weaknesses in clarity and completeness, allowing the revised version to address them.

## Are these controls enough?

No. Automated evaluation metrics are valuable, but human review is still necessary to verify factual accuracy, nuance, and context.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
